<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 2.5：融会贯通：一个 FIR 滤波器
**上一步：[时序逻辑](2.4_sequential_logic.ipynb)**<br>
**下一步：[ChiselTest (曾用名 chisel-testers2)](2.6_chiseltest.ipynb)**

## 动机
既然您已经学习了 Chisel 的基础知识，让我们利用这些知识来构建一个 FIR（有限脉冲响应）滤波器模块吧！FIR 滤波器在数字信号处理应用中非常常见。此外，FIR 滤波器将在模块 3 中频繁出现，因此请不要跳过本模块而忽略它！如果您不熟悉 FIR 滤波器，请访问 [可靠的维基百科](https://en.wikipedia.org/wiki/Finite_impulse_response) 了解更多信息。

## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test

---
# FIR 滤波器

您将设计的 FIR 滤波器执行以下操作。

<img src="images/fir.jpg" width="720">

基本上，这是将滤波器系数的元素与输入信号的元素进行逐元素乘法，并输出总和（也称为_卷积_）。

或者，信号定义如下：

$y[n] = b_0 x[n] + b_1 x[n-1] + b_2 x[n-2] + ...$
 - $y[n]$ 是时间 $n$ 的输出信号
 - $x[n]$ 是输入信号
 - $b_i$ 是滤波器系数或脉冲响应
 - $n-1$, $n-2$, ... 是时间 $n$ 延迟 1、2 ... 个周期
 
## 8 位规范

构建一个 4 元 FIR 滤波器，其中四个滤波器系数是参数。为您提供了一个模块骨架和基本测试。
请注意，输入和输出都是 8 位无符号整数。您需要使用移位寄存器等结构来保存必要的状态（例如延迟的信号值）。使用提供的测试程序来检查您的实现。
具有恒定输入的寄存器可以使用移位值为 1 的 `ShiftRegister` 或使用 `RegNext` 结构来创建。

注意：为了通过测试，您的寄存器必须初始化为 `0.U`。

In [ ]:
class My4ElementFir(b0: Int, b1: Int, b2: Int, b3: Int) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(8.W))
    val out = Output(UInt(8.W))
  })

  ???
}

In [ ]:
// 简单健全性检查：所有系数均为零的元素应始终产生零
test(new My4ElementFir(0, 0, 0, 0)) { c =>
    c.io.in.poke(0.U)
    c.io.out.expect(0.U)
    c.clock.step(1)
    c.io.in.poke(4.U)
    c.io.out.expect(0.U)
    c.clock.step(1)
    c.io.in.poke(5.U)
    c.io.out.expect(0.U)
    c.clock.step(1)
    c.io.in.poke(2.U)
    c.io.out.expect(0.U)
}

In [ ]:
// 简单的 4 点移动平均
test(new My4ElementFir(1, 1, 1, 1)) { c =>
    c.io.in.poke(1.U)
    c.io.out.expect(1.U)  // 1, 0, 0, 0
    c.clock.step(1)
    c.io.in.poke(4.U)
    c.io.out.expect(5.U)  // 4, 1, 0, 0
    c.clock.step(1)
    c.io.in.poke(3.U)
    c.io.out.expect(8.U)  // 3, 4, 1, 0
    c.clock.step(1)
    c.io.in.poke(2.U)
    c.io.out.expect(10.U)  // 2, 3, 4, 1
    c.clock.step(1)
    c.io.in.poke(7.U)
    c.io.out.expect(16.U)  // 7, 2, 3, 4
    c.clock.step(1)
    c.io.in.poke(0.U)
    c.io.out.expect(12.U)  // 0, 7, 2, 3
}

In [ ]:
// 非对称滤波器
test(new My4ElementFir(1, 2, 3, 4)) { c =>
    c.io.in.poke(1.U)
    c.io.out.expect(1.U)  // 1*1, 0*2, 0*3, 0*4
    c.clock.step(1)
    c.io.in.poke(4.U)
    c.io.out.expect(6.U)  // 4*1, 1*2, 0*3, 0*4
    c.clock.step(1)
    c.io.in.poke(3.U)
    c.io.out.expect(14.U)  // 3*1, 4*2, 1*3, 0*4
    c.clock.step(1)
    c.io.in.poke(2.U)
    c.io.out.expect(24.U)  // 2*1, 3*2, 4*3, 1*4
    c.clock.step(1)
    c.io.in.poke(7.U)
    c.io.out.expect(36.U)  // 7*1, 2*2, 3*3, 4*4
    c.clock.step(1)
    c.io.in.poke(0.U)
    c.io.out.expect(32.U)  // 0*1, 7*2, 2*3, 3*4
}

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  val x_n1 = RegNext(io.in, 0.U)
  val x_n2 = RegNext(x_n1, 0.U)
  val x_n3 = RegNext(x_n2, 0.U)
  io.out := io.in * b0.U(8.W) + 
    x_n1 * b1.U(8.W) +
    x_n2 * b2.U(8.W) + 
    x_n3 * b3.U(8.W)
</pre></article></div></section></div>

---
# FIR 滤波器生成器

对于本模块，我们将使用一个略微修改过的示例，该示例来自[模块 3.2：生成器：集合](3.2_collections.ipynb)。
如果您还没有开始学习模块 3.2，请不要担心。
您将了解 `MyManyDynamicElementVecFir` 工作原理的详细信息，但其基本思想是它是一个 FIR 滤波器生成器。

该生成器有一个参数：长度。
该参数决定了滤波器有多少个抽头，这些抽头是硬件 `Module` 的输入。

该生成器有 3 个输入：
* in，滤波器的输入
* valid，一个布尔值，指示输入何时有效
* consts，一个包含所有抽头的向量

和 1 个输出：
* out，滤波后的输入

<img src="images/fir.jpg" style="width:450px;"/>

In [ ]:
class MyManyDynamicElementVecFir(length: Int) extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(8.W))
    val valid = Input(Bool())
    val out = Output(UInt(8.W))
    val consts = Input(Vec(length, UInt(8.W)))
  })
  
  // 如此简洁！稍后您将了解所有这些的含义。
  val taps = Seq(io.in) ++ Seq.fill(io.consts.length - 1)(RegInit(0.U(8.W)))
  taps.zip(taps.tail).foreach { case (a, b) => when (io.valid) { b := a } }

  io.out := taps.zip(io.consts).map { case (a, b) => a * b }.reduce(_ + _)
}

visualize(() => new MyManyDynamicElementVecFir(4))

---
# DspBlock

将 DSP 组件集成到更大的系统中可能具有挑战性且容易出错。
[dsptools 存储库的 rocket 部分](https://github.com/ucb-bar/dsptools/tree/master/rocket) 包含一些有用的生成器，应该有助于完成此类任务。

核心抽象之一是 `DspBlock` 的概念。
`DspBlock` 具有：
* AXI-4 流输入和输出
* 内存映射状态和控制（在此示例中为 AXI4）

<img src="images/fir_filter.png" style="width:800px;"/>

`DspBlock` 使用来自 rocket 的外交接口。
[此站点](https://www.lowrisc.org/docs/diplomacy/) 对外交的基本知识进行了很好的概述，但对于此示例，不必过于担心它的工作原理。
当您将许多不同的块连接在一起以形成复杂的 SoC 时，外交的优势才能真正体现出来。
在此示例中，我们只是制作一个外围设备。
`StandaloneBlock` trait 被混入以使外交接口作为顶级 IO 工作。
仅当 `DspBlock` 用作顶级接口且没有任何外交连接时才需要它们。

以下代码将 FIR 滤波器封装在 AXI4 接口中。

In [ ]:
import dspblocks._
import freechips.rocketchip.amba.axi4._
import freechips.rocketchip.amba.axi4stream._
import freechips.rocketchip.config._
import freechips.rocketchip.diplomacy._
import freechips.rocketchip.regmapper._

//
// 所有 FIRBlock 的基类。
// 可以扩展此类以制作 TileLink、AXI4、APB、AHB 等风格的 FIR 滤波器
//
abstract class FIRBlock[D, U, EO, EI, B <: Data](val nFilters: Int, val nTaps: Int)(implicit p: Parameters)
// HasCSR 表示内存接口将使用 RegMapper API 来定义状态和控制寄存器
extends DspBlock[D, U, EO, EI, B] with HasCSR {
    // 流接口的外交节点
    // 身份节点表示输出和输入的参数化相同
    val streamNode = AXI4StreamIdentityNode()
    
    // 定义将要细化的硬件
    lazy val module = new LazyModuleImp(this) {
        // 从外交节点获取流输入和输出线
        val (in, _)  = streamNode.in(0)
        val (out, _) = streamNode.out(0)

        require(in.params.n >= nFilters,
                s"""AXI-4 Stream 端口必须足够大以容纳所有
                   |滤波器 (需要 $nFilters 个，但只有 ${in.params.n} 个)""".stripMargin)

        // 创建寄存器以存储抽头
        val taps = Reg(Vec(nFilters, Vec(nTaps, UInt(8.W))))

        // 内存映射抽头，第一个地址是一个只读字段，指示有多少个滤波器通道
        val mmap = Seq(
            RegField.r(64, nFilters.U, RegFieldDesc("nFilters", "滤波器通道数"))
        ) ++ taps.flatMap(_.map(t => RegField(8, t, RegFieldDesc("tap", "抽头"))))

        // 为内存接口生成硬件
        // 在此类中，regmap 是抽象的（未实现）。混入类似 AXI4HasCSR 或 TLHasCSR 的东西
        // 将为特定的内存接口定义 regmap
        regmap(mmap.zipWithIndex.map({case (r, i) => i * 8 -> Seq(r)}): _*)

        // 创建 FIR 通道并连接输入和抽头
        val outs = for (i <- 0 until nFilters) yield {
            val fir = Module(new MyManyDynamicElementVecFir(nTaps))
            
            fir.io.in := in.bits.data((i+1)*8, i*8)
            fir.io.valid := in.valid && out.ready
            fir.io.consts := taps(i)            
            fir.io.out
        }

        val output = if (outs.length == 1) {
            outs.head
        } else {
            outs.reduce((x: UInt, y: UInt) => Cat(y, x))
        }

        out.bits.data := output
        in.ready  := out.ready
        out.valid := in.valid
    }
}

// 创建 AXI4 风格的 FIRBlock
abstract class AXI4FIRBlock(nFilters: Int, nTaps: Int)(implicit p: Parameters) extends FIRBlock[AXI4MasterPortParameters, AXI4SlavePortParameters, AXI4EdgeParameters, AXI4EdgeParameters, AXI4Bundle](nFilters, nTaps) with AXI4DspBlock with AXI4HasCSR {
    override val mem = Some(AXI4RegisterNode(
        AddressSet(0x0, 0xffffL), beatBytes = 8
    ))
}

// 运行下面的代码将显示生成的 firrtl
// 注意 LazyModules 并不是真正的 chisel 模块——在调用 chisel 驱动程序时需要对它们调用“.module”
// 另请注意 AXI4StandaloneBlock 是混入的——如果忘记它，将会得到奇怪的外交错误，因为内存
// 接口期望一个主设备，而流接口期望被连接。AXI4StandaloneBlock 将添加顶层 IO
// println(chisel3.Driver.emit(() => LazyModule(new AXI4FIRBlock(1, 8)(Parameters.empty) with AXI4StandaloneBlock).module))

## 测试

测试 `DspBlock` 有点不同。
现在我们正在处理内存接口和 `LazyModule`。
dsptools 有一些功能可以帮助测试 `DspBlock`。

一个重要的功能是 `MemMasterModel`。
该 trait 定义了诸如 `memReadWord` 和 `memWriteWord` 之类的函数——用于生成内存流量的通用函数。
这允许您编写一个通用测试，该测试可以专门用于您正在使用的内存接口——例如，您编写一个测试，然后将其专门用于 TileLink 和 AXI4 接口。

下面的代码以这种方式测试 `FIRBlock`。

In [ ]:
import dsptools.tester.MemMasterModel
import freechips.rocketchip.amba.axi4

abstract class FIRBlockTester[D, U, EO, EI, B <: Data](c: FIRBlock[D, U, EO, EI, B]) extends PeekPokeTester(c.module) with MemMasterModel {
    // 检查地址 0 是否为滤波器数量
    require(memReadWord(0) == c.nFilters)
    // 将所有抽头写入 1
    for (i <- 0 until c.nFilters * c.nTaps) {
        memWriteWord(8 + i * 8, 1)
    }
}

// 为 axi4 特化通用测试器
class AXI4FIRBlockTester(c: AXI4FIRBlock with AXI4StandaloneBlock) extends FIRBlockTester(c) with AXI4MasterModel {
    def memAXI = c.ioMem.get
}

// 在 lazymodules 上调用测试器有点奇怪。
// 注意 firblocktester 接受一个 lazymodule，而不是一个 module（它在 "extends PeekPokeTester()" 中调用 .module）。
val lm = LazyModule(new AXI4FIRBlock(1, 8)(Parameters.empty) with AXI4StandaloneBlock)
chisel3.iotesters.Driver(() => lm.module) { _ => new AXI4FIRBlockTester(lm) }

<span style="color:red">**练习：TileLink**</span><br>

添加一个使用 TileLink 作为其内存互连的 `FIRBlock` 版本，并扩展 `FIRBlockTester` 以使用 TileLink。

---
# 您已完成！

[返回顶部。](#top)